In [0]:
from pyspark.sql import functions as f
from pyspark.sql.types import *
import re
from datetime import datetime

p_schema = StructType([
    StructField('product_id',IntegerType()),
    StructField('product_name',StringType()),
    StructField('category',StringType()),
    StructField('brand',StringType()),
    StructField('price',IntegerType()),
    StructField('status',StringType()),
])

product_files = []
files = dbutils.fs.ls('/Volumes/madstagescd2/madstage/madstage/incrementalLoad/')

for file in files:
    matched = re.search(r'(\d{4}-\d{2}-\d{2})',file.name)
    if file.name.startswith('product'):
        if matched:
            file_date = datetime.strptime(matched.group(1),"%Y-%m-%d")
            product_files.append((file_date,file.path))

if product_files:
    latest_files = max(product_files,key=lambda x:x[0])
    inc_prod_df = (spark.read.format('csv').option('header','true').schema(p_schema).load(latest_files[1]))

inc_prod_df = inc_prod_df.dropna(subset=('product_id'))
inc_prod_df = inc_prod_df.dropDuplicates()
inc_prod_df = inc_prod_df.fillna({
    'product_name':'Unknown',
    'category':'Unknown',
    'brand':'Unknown',
    'price':0,
    'status':'Unknown',
})

phone_regex = r'[^0-9]'
inc_prod_df = inc_prod_df.withColumn('product_name',f.initcap(f.trim(f.col('product_name')))).withColumn('category',f.initcap(f.trim(f.col('category')))).withColumn('brand',f.initcap(f.trim(f.col('brand')))).withColumn('price',f.round(f.col('price'),2)).withColumn('status',f.initcap(f.trim(f.col('status'))))

email_regex = r'^[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,}$'
email_validation =(f.col('email').rlike(email_regex))
phone_validation = (f.length(f.col('phone'))==10)

inc_prod_df = inc_prod_df.withColumn('dataValidation',f.when((f.col('status')=='Active')&(f.col('price')>=0),'Pass').otherwise('Fail'))

inc_prod_df = inc_prod_df.withColumn('is_current',f.lit('Y')).withColumn('updated_dt',f.lit(None).cast('string')).withColumn('created_dt',f.current_date().cast('string'))

inc_prod_df = inc_prod_df.filter(f.col('dataValidation')=='Yes').drop('dataValidation')

# Legacy 
legacy_df = (spark.read.table('madstagescd2.madstage.silver_products').filter(f.col('is_current')=='Y'))
inc_prod_df.createOrReplaceTempView('inc_prod_df')

spark.sql("""
        merge into madstagescd2.madstage.silver_products as t
        using inc_prod_df as s
        on t.product_id = s.product_id
        when matched then
        update set
        t.product_name = s.product_name,
        t.category = s.category,
        t.brand = s.brand,
        t.price = s.price,
        t.status = s.status,
        t.is_current = s.is_current,
        t.updated_dt = s.updated_dt,
        t.created_dt = s.created_dt
        when not matched then
        insert *
""")

print('SCD1 implemented successfully')

